author: baptisteD

# Sprint 1 — Ingestion : carte d'identité des fichiers sources

## Objectif

Avant d'exploiter les données, je veux savoir **ce que contient chaque fichier** : sa taille, son encodage, son nombre de lignes et de colonnes, le type de chaque colonne et quelques statistiques (min, max, moyenne, médiane).

Les sources sont dans `Datas/` :
- `releves_incidents.csv` : les relevés d'incidents saisis par les opérateurs,
- `telemetry.csv` : la télémétrie horaire des machines,
- `machine.sql` : un **script** PostgreSQL (référentiel machines + maintenance), donc pas un fichier tabulaire.

## Démarche générale

1. Plutôt que d'écrire le traitement directement dans le notebook, je l'ai mis dans des **classes Python** du package `src/indusense/`. Je peux ainsi les réutiliser sur n'importe quel fichier, les tester et les appeler depuis un script. Le notebook sert uniquement à les piloter et à expliquer la démarche.
2. Je traite d'abord les **CSV** avec `IdentityGenerator`.
3. Pour le **SQL**, je ne réécris pas un second analyseur : je **convertis le dump en CSV** avec `DumpToCsv`, puis je réutilise le même `IdentityGenerator`. Toutes les sources passent ainsi par le même traitement et produisent des cartes au même format.

### Import des classes

Le package est dans `src/`, je l'ajoute donc au `sys.path` pour pouvoir l'importer depuis le notebook.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("src").resolve()))

from indusense.identity_generator import IdentityGenerator
from indusense.dump_to_csv import DumpToCsv

## 1. La carte d'identité d'un CSV : `IdentityGenerator`

### Où sont écrits les fichiers ?

`artifacts/ingestion/identities/<YYYYMMDD>/<nom_source>_<HHMMSS>.md`

- un **dossier par jour** pour regrouper les exécutions d'une même journée,
- l'**heure dans le nom du fichier** pour ne jamais écraser une carte précédente : je garde l'historique et je peux comparer deux passages,
- du **Markdown**, lisible directement dans VS Code ou sur un dépôt Git.

### Mes choix pour chaque information

| Information | Comment je l'obtiens | Pourquoi |
| --- | --- | --- |
| **Source** | Chemin **relatif à la racine du projet** (le dossier qui contient `pyproject.toml`) | Un chemin absolu (`C:\wamp\...`) ne vaut que sur ma machine ; le chemin relatif reste valable pour toute l'équipe. |
| **Filesize** | Taille en octets convertie en o / Ko / Mo / Go | Plus lisible qu'un nombre d'octets brut. |
| **ProcessBeginAt / EndAt** | Heure avant la lecture / après l'analyse | Mesure le temps de traitement de chaque fichier. |
| **Charset** | J'essaie de décoder le fichier en `utf-8`, puis en `iso-8859-1` | UTF-8 est strict : s'il décode sans erreur, c'est le bon encodage. Sinon, c'est généralement un fichier Latin-1 (Excel sous Windows). Pas besoin de librairie supplémentaire. |
| **Rows / Cols** | `len(df)` et `len(df.columns)` après lecture par pandas | |
| **Séparateur** | Paramètre `separator` du constructeur (`,` par défaut) | Un CSV peut utiliser `,`, `;` ou la tabulation : je ne le code pas en dur. |

### Inférence du type des colonnes

Je teste les règles dans cet ordre :
1. nom qui se termine par `_id` → **IDENTITY** (un identifiant, même numérique, n'a pas de sens statistique),
2. type booléen pandas → **BOOLEAN**,
3. type numérique pandas → **NUMERIC**,
4. les 100 premières valeurs commencent par `AAAA-MM-JJ` → **DATE** (pandas lit les dates comme du texte, d'où ce test),
5. sinon → **TEXT**.

### Statistiques

- **NUMERIC** : min, max, moyenne, médiane,
- **DATE** : min et max uniquement (une moyenne de dates n'a pas d'intérêt ici),
- les autres types sont ignorés, comme le demande la consigne.

In [ ]:
generator = IdentityGenerator()  # séparateur "," par défaut

for csv in ["Datas/releves_incidents.csv", "Datas/telemetry.csv"]:
    print(generator.generateIdentities(csv))

## 2. Le cas du script SQL : `DumpToCsv`

### Le problème

`machine.sql` n'est pas un tableau : c'est un script qui contient des `BEGIN`, des `CREATE TABLE`, des `INSERT` et un `COMMIT`. Je ne peux pas le lire avec `pd.read_csv`, et le découper avec des expressions régulières serait fragile (virgules ou apostrophes dans les valeurs, `INSERT` sur plusieurs lignes…).

### La solution : un vrai parseur SQL

J'ai choisi **[sqlglot](https://github.com/tobymao/sqlglot)** (`uv add sqlglot`) :
- c'est du Python pur, sans base de données à installer,
- il connaît le **dialecte PostgreSQL** du dump (paramètre `dialect`),
- il transforme le script en **arbre syntaxique** (AST) : je manipule des objets `Insert`, `Values`, `Schema` au lieu de chaînes de caractères.

### Le raisonnement de `extractTables`

1. Je parse tout le script et je parcours les instructions une par une.
2. Je ne garde **que les `INSERT ... VALUES`**. `BEGIN`, `CREATE TABLE`, `COMMIT` ou `INSERT ... SELECT` ne contiennent pas de données à extraire.
3. Pour chaque `INSERT`, je récupère le **nom de la table** et, s'il y en a une, la **liste des colonnes**.
4. Chaque ligne de `VALUES` devient une ligne de DataFrame. Les littéraux SQL sont convertis en valeurs Python (`'abc'` → `str`, `12` → `int`, `TRUE` → `bool`, `NULL` → `None`). Pour une expression non littérale comme `NOW()`, je garde le texte SQL plutôt que de planter.
5. Une même table peut avoir **plusieurs `INSERT`** : je les **concatène** pour obtenir un seul DataFrame par table.

### Écriture

`convert` écrit **un CSV par table** dans `Datas/dump/<table>_<YYYYMMDD_HHMMSS>.csv`. Le séparateur et l'encodage sont paramétrables, comme pour `IdentityGenerator`.

In [ ]:
dump = DumpToCsv()  # dialecte postgres, sortie dans Datas/dump
dump_csvs = dump.convert("Datas/machine.sql")
dump_csvs

## 3. Carte d'identité des tables issues du SQL

Les tables sont maintenant des CSV classiques : je réutilise exactement le même générateur.

In [ ]:
for csv in dump_csvs:
    print(generator.generateIdentities(csv))

## Limites et pistes d'amélioration

- **Détection de l'encodage** : seuls UTF-8 et ISO-8859-1 sont testés. Pour d'autres encodages, une librairie comme `charset-normalizer` serait plus fiable.
- **Inférence des types** : la règle `_id` et le test de date sur 100 valeurs sont des heuristiques simples. Une colonne comme `operator_badge` reste en TEXT alors que c'est aussi un identifiant.
- **Séparateur** : il est fixé pour toute l'instance ; on pourrait le détecter automatiquement avec `csv.Sniffer`.
- **Mémoire** : le CSV est chargé entièrement. Pour de très gros fichiers, il faudrait le lire par blocs (`chunksize`).
- **SQL** : les types déclarés dans les `CREATE TABLE` ne sont pas encore exploités. Ils pourraient remplacer l'inférence pour les tables issues du dump.